<a href="https://colab.research.google.com/github/Jasonmiller513/Dissertation/blob/Updates/First%20Test%20WA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **STEP 1: CLEAN DATA**

# Import Extensions And Open File

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
import joblib
from collections import defaultdict
df = pd.read_csv('DataCoSupplyChainDataset.csv', encoding='latin1')

# Remove Unnecessary Columns

In [2]:
# List of columns to remove
columns_to_remove = [

    # Identifier columns
    'Customer Id',
    'Category Id',
    'Department Id',

    # Personal/customer detail columns
    'Customer Email',
    'Customer Password',
    'Customer Fname',
    'Customer Lname',
    'Customer Street',
    'Customer Zipcode',
    'Customer Full Name',

    # High-cardinality text columns
    'Product Description',
    'Product Image',
    'Product Name',


    # Duplicate or Redundant Geographic Columns

'Customer Segment',
'Order City',
'Order State',]

# Remove only columns that actually exist in dataframe
existing_columns_to_remove = [
    col for col in columns_to_remove if col in df.columns]

# Drop columns
df = df.drop(columns=existing_columns_to_remove)

# Display remaining columns
print("Remaining Columns:")
print(df.columns.tolist())

# Display shape after removal
print("\nNew Dataset Shape:")
print(df.shape)

Remaining Columns:
['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Name', 'Customer City', 'Customer Country', 'Customer State', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order Country', 'Order Customer Id', 'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order Status', 'Order Zipcode', 'Product Card Id', 'Product Category Id', 'Product Price', 'Product Status', 'shipping date (DateOrders)', 'Shipping Mode']

New Dataset Shape:
(180519, 38)


# Identify missing or erroneous data


In [3]:

print(df.shape)
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0])

(180519, 38)
Order Zipcode    155679
dtype: int64


In [4]:
duplicate_count = df.duplicated().sum()
print(f"Duplicate Rows: {duplicate_count}")

Duplicate Rows: 0


In [5]:
missing = df.isnull().sum()
print(missing[missing > 0])

Order Zipcode    155679
dtype: int64


In [6]:
num_cols = df.select_dtypes(include=['int64','float64']).columns

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())
    cat_cols = df.select_dtypes(include=['object']).columns

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [7]:
non_negative_columns = [
    'Sales',
    'Order Item Quantity',
    'Order Item Total',
    'Product Price',
    'Days for shipping (scheduled)']

for col in non_negative_columns:

    if col in df.columns:

        # Count invalid negatives
        invalid_count = (df[col] < 0).sum()

        print(f"\n{col} Negative Values: {invalid_count}")

        # Replace negatives with median
        median_value = df[df[col] >= 0][col].median()

        df.loc[df[col] < 0, col] = median_value


Sales Negative Values: 0

Order Item Quantity Negative Values: 0

Order Item Total Negative Values: 0

Product Price Negative Values: 0


In [8]:
numeric_cols = df.select_dtypes(include=np.number).columns

outlier_summary = {}

for col in numeric_cols:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = ((df[col] < lower) | (df[col] > upper)).sum()

    outlier_summary[col] = outliers

outlier_df = pd.DataFrame.from_dict(
    outlier_summary,
    orient='index',
    columns=['Outlier Count']
)

print(outlier_df.sort_values('Outlier Count', ascending=False))

                               Outlier Count
Order Zipcode                          24818
Benefit per order                      18942
Order Profit Per Order                 18942
Order Item Profit Ratio                17300
Order Item Discount                     7537
Order Item Product Price                2048
Product Price                           2048
Sales per customer                      1943
Order Item Total                        1943
Longitude                               1414
Order Customer Id                       1198
Sales                                    488
Latitude                                   9
Late_delivery_risk                         0
Days for shipment (scheduled)              0
Days for shipping (real)                   0
Order Item Quantity                        0
Order Item Id                              0
Order Item Cardprod Id                     0
Order Item Discount Rate                   0
Order Id                                   0
Product Ca

# Winsorize Outliers

In [9]:
for col in numeric_cols:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = np.where(df[col] < lower, lower, df[col])
    df[col] = np.where(df[col] > upper, upper, df[col])

# Log Transform Highly Skewed Variables

In [10]:
skew_cols = [
    'Order Item Total',
    'Order Profit Per Order',
    'Shipping Cost',
    'Profit Per Unit'
]

for col in skew_cols:
    if col in df.columns:
        df[col] = np.log1p(df[col] - df[col].min())

In [11]:
print("\nRemaining Missing Values:")
print(df.isnull().sum().sum())

print("\nFinal Dataset Shape:")
print(df.shape)

print("\nDataset Preview:")
print(df.head())


Remaining Missing Values:
0

Final Dataset Shape:
(180519, 38)

Dataset Preview:
       Type  Days for shipping (real)  Days for shipment (scheduled)  \
0     DEBIT                       3.0                            4.0   
1  TRANSFER                       5.0                            4.0   
2      CASH                       4.0                            4.0   
3     DEBIT                       3.0                            4.0   
4   PAYMENT                       2.0                            4.0   

   Benefit per order  Sales per customer   Delivery Status  \
0          91.250000          314.640015  Advance shipping   
1         -79.700005          311.359985     Late delivery   
2         -79.700005          309.720001  Shipping on time   
3          22.860001          304.809998  Advance shipping   
4         134.210007          298.250000  Advance shipping   

   Late_delivery_risk   Category Name Customer City Customer Country  ...  \
0                 0.0  Sporting Goo

# Remove Constant and Near-Constant Features

In [12]:
constant_cols = [
    col for col in df.columns
    if df[col].nunique() == 1
]

print("Constant Features:")
print(constant_cols)

df.drop(columns=constant_cols, inplace=True)
near_constant_cols = []

for col in df.columns:

    freq = df[col].value_counts(normalize=True)

    # Check if freq is empty (e.g., column was entirely NaN) or if a single value dominates
    if freq.empty or freq.iloc[0] > 0.99:
        near_constant_cols.append(col)

print("Near Constant Features:")
print(near_constant_cols)

df.drop(columns=near_constant_cols, inplace=True)

Constant Features:
['Order Zipcode', 'Product Status']
Near Constant Features:
[]


# Remove Duplicate Rows

In [13]:
print("Rows before:", len(df))

df.drop_duplicates(inplace=True)

print("Rows after:", len(df))

Rows before: 180519
Rows after: 180519


# Verify Dataset Quality

In [14]:
print("Remaining Missing Values:")
print(df.isnull().sum().sum())

print("Dataset Shape:")
print(df.shape)

Remaining Missing Values:
0
Dataset Shape:
(180519, 36)


# Recompute Numeric Columns

In [15]:
numeric_cols = df.select_dtypes(include=['int64','float64']).columns.tolist()

print(f"Number of numeric features: {len(numeric_cols)}")
print(numeric_cols)

Number of numeric features: 22
['Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Late_delivery_risk', 'Latitude', 'Longitude', 'Order Customer Id', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Product Card Id', 'Product Category Id', 'Product Price']


#Save Data

In [16]:
# Save cleaned RL dataset to CSV in Google Colab

output_file = 'dataco_rl_cleaned.csv'

# Save dataframe
df.to_csv(output_file, index=False)

print(f"Dataset saved as: {output_file}")

Dataset saved as: dataco_rl_cleaned.csv


# **STEP 2: FEATURE ENGINEERING**

In [17]:
df = pd.read_csv('dataco_rl_cleaned.csv', encoding='latin1')

print("Dataset Loaded")
print(df.shape)

Dataset Loaded
(180519, 36)


# Date and Time Features

In [18]:
# Convert order date to datetime
df['order date (DateOrders)'] = pd.to_datetime(
    df['order date (DateOrders)'],
    errors='coerce')

# Extract temporal features
df['order_year'] = df['order date (DateOrders)'].dt.year
df['order_month'] = df['order date (DateOrders)'].dt.month
df['order_day'] = df['order date (DateOrders)'].dt.day
df['order_dayofweek'] = df['order date (DateOrders)'].dt.dayofweek
df['is_weekend'] = df['order_dayofweek'].isin([5,6]).astype(int)


# Create a Product-Region Matrix and Network

In [19]:
# Create a Product-Region Matrix
product_region = (
    df.groupby("Product Category Id")["Order Region"]
      .nunique()
      .reset_index(name="num_regions")
      .sort_values("num_regions", ascending=False))

In [20]:
order_features = (
    df.groupby("Order Id")
      .agg(
          unique_products=("Product Category Id", "nunique"),
          total_quantity=("Order Item Quantity", "sum"),
          total_sales=("Sales", "sum"))
      .reset_index())

order_features["complex_order"] = (
    order_features["unique_products"] >= 3
).astype(int)

In [21]:
network = (
    df.groupby(["Product Category Id", "Order Region"])
      .agg(
          orders=("Order Id", "nunique"),
          quantity=("Order Item Quantity", "sum"),
          revenue=("Sales", "sum"))
      .reset_index())

# Explore Product Statistics

In [22]:
product_stats = (
    network.groupby("Product Category Id")
           .agg(
               regions_served=("Order Region", "nunique"),
               total_orders=("orders", "sum"),
               total_quantity=("quantity", "sum"),
               total_revenue=("revenue", "sum"))
           .reset_index())


# Create Shipping Mode Statistics

In [23]:
import pandas as pd
import numpy as np

# Ensure 'Shipping Mode_str' and 'Order Region_str' are available
df['Shipping Mode_str'] = df['Shipping Mode']
df['Order Region_str'] = df['Order Region']

df = df[df['Order Region_str'] == 'West Africa'].copy()

# Historical shipping mode performance
mode_stats = (
    df.groupby('Shipping Mode_str')
      .agg({
          'Order Profit Per Order':'mean',
          'Late_delivery_risk':'mean',
          'Days for shipping (real)':'mean'
      })
      .reset_index()
)

print(mode_stats)

  Shipping Mode_str  Order Profit Per Order  Late_delivery_risk  \
0       First Class                4.368462            0.978402   
1          Same Day                4.236680            0.354260   
2      Second Class                4.310360            0.772358   
3    Standard Class                4.396597            0.374560   

   Days for shipping (real)  
0                  2.000000  
1                  0.390135  
2                  3.979675  
3                  4.010563  


# Define Multi-Region Stock

In [24]:
region_threshold = product_stats["regions_served"].median()
order_threshold = product_stats["total_orders"].median()

product_stats["likely_multi_region_stocked"] = (
    (product_stats["regions_served"] >= region_threshold)
    &
    (product_stats["total_orders"] >= order_threshold)
).astype(int)

#Create a stock score

In [25]:
product_stats["stocking_score"] = (
    0.5 *
    (
        product_stats["regions_served"]
        / product_stats["regions_served"].max())
    +
    0.5 *
    (
        product_stats["total_orders"]
        / product_stats["total_orders"].max()))


# Merge the network

In [26]:
network = network.merge(
    product_stats[
        [
            "Product Category Id",
            "regions_served",
            "stocking_score",
            "likely_multi_region_stocked"
        ]
    ],
    on="Product Category Id",
    how="left")

# Find Edge Weight

In [27]:
network["edge_weight"] = (
    0.4 *
    (network["orders"] / network["orders"].max())
    +
    0.3 *
    (network["quantity"] / network["quantity"].max())
    +
    0.3 *
    (network["revenue"] / network["revenue"].max()))


 # Find candidate shipping regions

In [28]:
candidate_regions = (
    network.sort_values(
        ["Product Category Id", "edge_weight"],
        ascending=[True, False])
    .groupby("Product Category Id")
    .head(5))

# Route Variable Creation

In [30]:
import pandas as pd

# Load the dataframes if they are not defined.
# This ensures the cell can run independently if necessary.
try:
    train_df.head() # Check if train_df exists
except NameError:
    train_df = pd.read_csv("dataco_rl_train_discrete.csv")
    val_df = pd.read_csv("dataco_rl_validation.csv")
    test_df = pd.read_csv("dataco_rl_test_discrete.csv")

# Ensure 'Order Region_str' is present after loading. This is also handled in NA954ND-_EH5.
for d in [train_df, val_df, test_df]:
    if 'Order Region_str' not in d.columns:
        # Assuming 'West Africa' from previous filtering context
        d['Order Region_str'] = 'West Africa'

# Route definition
train_df["Route"] = (
    train_df["Order Region_str"].astype(str) # Use Order Region_str instead of Order Region
    + " -> "
    + train_df["Order Country"].astype(str)
)

val_df["Route"] = (
    val_df["Order Region_str"].astype(str) # Use Order Region_str instead of Order Region
    + " -> "
    + val_df["Order Country"].astype(str)
)

test_df["Route"] = (
    test_df["Order Region_str"].astype(str) # Use Order Region_str instead of Order Region
    + " -> "
    + test_df["Order Country"].astype(str)
)

# Save Outputs

In [31]:
network.to_csv(
    "product_region_network.csv",
    index=False
)

candidate_regions.to_csv(
    "candidate_fulfillment_regions.csv",
    index=False
)

product_stats.to_csv(
    "product_stocking_scores.csv",
    index=False
)

print("Files created successfully")

Files created successfully


# Shipping Engineering

In [32]:
df['order date (DateOrders)'] = pd.to_datetime(
    df['order date (DateOrders)'],
    errors='coerce'
)
df['shipping date (DateOrders)'] = pd.to_datetime(
    df['shipping date (DateOrders)'],
    errors='coerce'
)

# Actual shipping time

df['actual_ship_days'] = (
    df['shipping date (DateOrders)']
    - df['order date (DateOrders)']
).dt.days

# Difference from scheduled

df['shipping_delay'] = (
    df['Days for shipping (real)']
    - df['Days for shipment (scheduled)']
)

# Binary on-time flag

df['on_time_delivery'] = (
    df['shipping_delay'] <= 0
).astype(int)

# Late shipment flag

df['late_delivery'] = (
    df['shipping_delay'] > 0
).astype(int)

# Profit Engineering

In [33]:
# Margin percentage

df['margin_pct'] = (
    df['Order Profit Per Order']
    / df['Sales']
)

# Profit per item

df['profit_per_item'] = (
    df['Order Profit Per Order']
    / df['Order Item Quantity']
)

# Discount percentage

df['discount_pct'] = (
    df['Order Item Discount']
    / df['Order Item Product Price']
)

# Product Demand Features

In [34]:
product_stats = (
    df.groupby('Product Category Id')
      .agg(
          product_orders=('Order Id','nunique'),
          product_quantity=('Order Item Quantity','sum'),
          product_revenue=('Sales','sum')
      )
      .reset_index()
)

df = df.merge(
    product_stats,
    on='Product Category Id',
    how='left'
)

# Regional Features

In [35]:
region_stats = (
    df.groupby('Order Region')
      .agg(
          region_orders=('Order Id','nunique'),
          region_sales=('Sales','sum'),
          region_profit=('Order Profit Per Order','sum')
      )
      .reset_index()
)

df = df.merge(
    region_stats,
    on='Order Region',
    how='left'
)

# Fulfillment Network Features

In [36]:
stocking = pd.read_csv(
    "product_stocking_scores.csv"
)

df = df.merge(
    stocking[
        [
            'Product Category Id',
            'stocking_score',
            'regions_served',
            'likely_multi_region_stocked'
        ]
    ],
    on='Product Category Id',
    how='left'
)

# Shipping Mode Features

In [37]:
speed_mapping = {
    'Same Day':4,
    'First Class':3,
    'Second Class':2,
    'Standard Class':1
}
df['shipping_mode_speed'] = df['Shipping Mode'].map(speed_mapping)

# Create string versions of Shipping Mode and Order Region for later use
df['Shipping Mode_str'] = df['Shipping Mode']
df['Order Region_str'] = df['Order Region']

action_id_mapping = {
    'Standard Class': 0,    # Maps to key 0 in shipping_cost_map
    'Second Class': 1,     # Maps to key 1 in shipping_cost_map
    'First Class': 2,      # Maps to key 2 in shipping_cost_map
    'Same Day': 3          # Maps to key 3 in shipping_cost_map
}
df['action_id'] = df['Shipping Mode'].map(action_id_mapping)

shipping_cost_map = {
    0: 1.0,    # Standard
    1: 1.5,    # Second
    2: 2.0,    # First
    3: 3.0     # Same Day
}

BASE_COST = 10

df['Mode_Cost'] = (
    BASE_COST *
    df['action_id'].map(shipping_cost_map)
)

# Order Complexity Features

In [38]:
order_complexity = (
    df.groupby('Order Id')
      .agg(
          unique_products=('Product Category Id','nunique'),
          total_quantity=('Order Item Quantity','sum'),
          total_sales=('Sales','sum')
      )
      .reset_index()
)

df = df.merge(
    order_complexity,
    on='Order Id',
    how='left'
)

df['complex_order'] = (
    df['unique_products'] >= 3
).astype(int)

# Route-Based Shipping Costs

In [39]:
df = pd.read_csv('dataco_rl_feature_engineered.csv')

# Ensure 'Shipping Cost' is present, assuming it should align with 'Mode_Cost'
df['Shipping Cost'] = df['Mode_Cost']

route_cost = (
    df.groupby(
        ['Order Region','Shipping Mode']
    )['Shipping Cost']
    .mean()
    .reset_index()
)

route_cost.rename(
    columns={
        'Shipping Cost':'route_cost'
    },
    inplace=True
)

df = df.merge(
    route_cost,
    on=['Order Region','Shipping Mode'],
    how='left'
)

# RL State Features

In [40]:
state_features = [
    'Product Category Id',
    'Order Region',
    'Market',
    'Shipping Mode',
    'stocking_score',
    'regions_served',
    'likely_multi_region_stocked',
    'unique_products',
    'total_quantity',
    'shipping_delay',
    'margin_pct',
    'profit_per_item',
    'order_month',
    'order_dayofweek'
]

# Reward Features

In [41]:
df['reward'] = (
    df['Order Profit Per Order']
    + 100 * df['on_time_delivery']
    - 20 * df['late_delivery']
)
LATE_PENALTY = 10

df['reward'] = (
    df['Order Profit Per Order']
    - df['Mode_Cost']
    - LATE_PENALTY * df['Late_delivery_risk']
)
df['net_profit'] = (
    df['Order Profit Per Order']
    - df['Shipping Cost']
)

# Save Engineered Dataset

In [42]:
df.to_csv(
    "dataco_rl_feature_engineered.csv",
    index=False
)

print(df.shape)

(3696, 72)


# **STEP 3: ENCODING CATEGORICAL VARIABLES**

In [43]:

df = pd.read_csv('dataco_rl_feature_engineered.csv', encoding='latin1')


print("Dataset Loaded")
print(df.shape)

Dataset Loaded
(3696, 72)


In [44]:
one_hot_cols = [
    'Shipping Mode',
    'Market',
    'Order Region'
]

df = pd.get_dummies(
    df,
    columns=one_hot_cols,
    drop_first=False
)

df.to_csv(
    "dataco_rl_onehot.csv",
    index=False
)

print(df.shape)

(3696, 75)


# Filter on West Africa

In [45]:
df = pd.read_csv("dataco_rl_onehot.csv")

# Keep only West Africa orders
west_africa_df = df[df["Order Region_West Africa"] == 1].copy()

print("Number of West Africa orders:", len(west_africa_df))

Number of West Africa orders: 3696


# **STEP 4: TRAIN/TEST TEMPORAL SPLIT**

In [46]:
import pandas as pd

df = pd.read_csv("dataco_rl_onehot.csv")

df = df[df["Order Region_West Africa"] == 1].copy()

df['order date (DateOrders)'] = pd.to_datetime(
    df['order date (DateOrders)']
)

df = df.sort_values(
    'order date (DateOrders)'
)

n = len(df)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df.iloc[:train_end]
val_df = df.iloc[train_end:val_end]
test_df = df.iloc[val_end:]

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(2587, 75)
(554, 75)
(555, 75)


In [47]:
# Save datasets

train_df.to_csv(
    "dataco_rl_train.csv",
    index=False
)

val_df.to_csv(
    "dataco_rl_validation.csv",
    index=False
)

test_df.to_csv(
    "dataco_rl_test.csv",
    index=False
)

print("Files saved successfully")

Files saved successfully


# STEP 5: STATE-SPACE DISCRETIZATION

# Feature Selection

In [48]:
state_features = [
    'Profit Margin',
    'Shipping Cost Ratio',
    'Historical On-Time Rate',
    'Regional Delay Score',
    'Product Category Id'
]

# Fit Bins Using Training Data Only

In [49]:
train_df = pd.read_csv("dataco_rl_train.csv")

# Use train_df instead of df for binning operations
train_df['shipping_cost_bin'] = pd.qcut(
    train_df['Shipping Cost'],
    q=5,
    labels=False,
    duplicates='drop'
)

train_df['order_value_bin'] = pd.qcut(
    train_df['Order Item Total'],
    q=5,
    labels=False,
    duplicates='drop'
)

# The 'Order Region' column does not exist in train_df.
# Since train_df is already filtered to 'West Africa' and 'Order Region'
# was one-hot encoded, 'Order Region_bin' will be a constant.
# Assigning a constant value to create the expected column.
train_df['Order Region_bin'] = 0

exclude_cols = [
    'Reward',
    'Late_delivery_risk',
    'Action_ID'
]

continuous_features = [
    col for col in train_df.select_dtypes(include='number').columns
    if col not in exclude_cols
    and not col.endswith('_bin') # Exclude the newly created bin columns from being binned again
]

bin_edges = {}

# Create 10 bins for each feature
for col in continuous_features:
    # Handle cases where all values are the same to prevent qcut errors
    if train_df[col].nunique() == 1:
        # If all values are the same, create a single bin from min to min (or min to min + epsilon)
        min_val = train_df[col].min()
        bins = np.array([min_val, min_val + 1e-9]) # Add a small epsilon to ensure two distinct points
    else:
        _, bins = pd.qcut(
            train_df[col],
            q=10,
            labels=False,
            retbins=True,
            duplicates='drop'
        )
    bin_edges[col] = bins

# Save bin definitions
joblib.dump(
    bin_edges,
    "state_bins.pkl"
)

print("Bins created successfully")

Bins created successfully


In [50]:
bin_counts = {}

for col in continuous_features:
    _, bins = pd.qcut(
        train_df[col],
        q=10,
        retbins=True,
        duplicates='drop'
    )

    bin_counts[col] = len(bins) - 1

print(bin_counts)

{'Days for shipping (real)': 5, 'Days for shipment (scheduled)': 3, 'Benefit per order': 10, 'Sales per customer': 10, 'Latitude': 10, 'Longitude': 10, 'Order Customer Id': 10, 'Order Id': 10, 'Order Item Cardprod Id': 9, 'Order Item Discount': 10, 'Order Item Discount Rate': 10, 'Order Item Id': 10, 'Order Item Product Price': 8, 'Order Item Profit Ratio': 10, 'Order Item Quantity': 4, 'Sales': 9, 'Order Item Total': 10, 'Order Profit Per Order': 10, 'Product Card Id': 9, 'Product Category Id': 9, 'Product Price': 8, 'order_year': 0, 'order_month': 4, 'order_day': 10, 'order_dayofweek': 6, 'is_weekend': 1, 'shipping_mode_speed': 3, 'action_id': 3, 'Mode_Cost': 3, 'actual_ship_days': 5, 'shipping_delay': 5, 'on_time_delivery': 1, 'late_delivery': 1, 'margin_pct': 10, 'profit_per_item': 10, 'discount_pct': 10, 'product_orders': 8, 'product_quantity': 9, 'product_revenue': 8, 'region_orders': 0, 'region_sales': 0, 'region_profit': 0, 'stocking_score': 8, 'regions_served': 1, 'likely_mult

# Apply Bins to Train/Test Data

In [51]:
train_df = pd.read_csv("dataco_rl_train.csv")
test_df = pd.read_csv("dataco_rl_test.csv")

bin_edges = joblib.load("state_bins.pkl")

continuous_features = list(bin_edges.keys())

for col in continuous_features:

    train_df[col + "_bin"] = np.digitize(
        train_df[col],
        bin_edges[col][1:-1]
    )

    test_df[col + "_bin"] = np.digitize(
        test_df[col],
        bin_edges[col][1:-1]
    )

train_df.to_csv(
    "dataco_rl_train_discrete.csv",
    index=False
)

test_df.to_csv(
    "dataco_rl_test_discrete.csv",
    index=False
)

print("Discretization complete")

Discretization complete


# Build RL States

In [52]:
import pandas as pd
import numpy as np

# Load the discretized training data
df = pd.read_csv("dataco_rl_train_discrete.csv")

# All data is already filtered for 'West Africa', so 'Order Region_bin' is a constant
df['Order Region_bin'] = 0

state_cols = [
    'Product Category Id_bin',
    'stocking_score_bin',
    'margin_pct_bin',
    'shipping_delay_bin',
    'Order Region_bin',
    'Order Item Total_bin', # Corrected name from 'order_value_bin'
    'Shipping Cost_bin'     # Corrected name from 'shipping_cost_bin'
]

df['state'] = (
    df[state_cols]
      .astype(str)
      .agg('_'.join, axis=1)
)

state_map = {
    s:i for i,s in enumerate(df['state'].unique())
}

df['state_id'] = df['state'].map(state_map)

num_states = len(state_map)

print("States:", num_states)

States: 1011


# Create State IDs

In [53]:
train_df = pd.read_csv("dataco_rl_train_discrete.csv")

state_cols = [
    'Product Category Id_bin',
    'stocking_score_bin',
    'margin_pct_bin',
    'shipping_delay_bin'
]

train_df['state'] = (
    train_df[state_cols]
      .astype(str)
      .agg('_'.join, axis=1)
)

state_encoder = LabelEncoder()

train_df['state_id'] = state_encoder.fit_transform(
    train_df['state']
)

joblib.dump(
    state_encoder,
    "state_encoder.pkl"
)

train_df['state_id'] = (
    train_df['state_id']
    .astype(int)
)

train_df['action_id'] = (
    train_df['action_id']
    .astype(int)
)

print(
    "Unique states:",
    train_df['state_id'].nunique()
)

Unique states: 377


# Create Action IDs

In [54]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import joblib

# Load split files
train_df = pd.read_csv("dataco_rl_train_discrete.csv")
val_df = pd.read_csv("dataco_rl_validation.csv")
test_df = pd.read_csv("dataco_rl_test_discrete.csv")

# Reload candidate fulfillment regions
candidate_regions = pd.read_csv("candidate_fulfillment_regions.csv")

candidate_regions = candidate_regions.rename(
    columns={"Order Region": "Fulfillment_Region"}
)

candidate_regions = candidate_regions[
    ["Product Category Id", "Fulfillment_Region"]
].drop_duplicates()

# Merge fulfillment region back in
train_df = train_df.merge(candidate_regions, on="Product Category Id", how="left")
val_df = val_df.merge(candidate_regions, on="Product Category Id", how="left")
test_df = test_df.merge(candidate_regions, on="Product Category Id", how="left")

# Restore string columns if needed
if "Shipping Mode_str" not in train_df.columns:
    shipping_mode_map_reverse = {
        1: "Standard Class",
        2: "Second Class",
        3: "First Class",
        4: "Same Day"
    }

    for d in [train_df, val_df, test_df]:
        d["Shipping Mode_str"] = d["shipping_mode_speed"].map(shipping_mode_map_reverse)

if "Order Region_str" not in train_df.columns:
    for d in [train_df, val_df, test_df]:
        d["Order Region_str"] = "West Africa"

# Create route
for d in [train_df, val_df, test_df]:
    d["Route"] = (
        d["Order Region_str"].astype(str)
        + " -> "
        + d["Order Country"].astype(str)
    )

# Combined action = Fulfillment Region + Route + Shipping Mode
for d in [train_df, val_df, test_df]:
    d["action"] = (
        d["Fulfillment_Region"].astype(str)
        + " | "
        + d["Route"].astype(str)
        + " | "
        + d["Shipping Mode_str"].astype(str)
    )

# Create states
state_cols = [
    "Product Category Id_bin",
    "stocking_score_bin",
    "margin_pct_bin",
    "shipping_delay_bin",
    "Shipping Cost_bin",
    "Order Item Total_bin"
]

train_df["state"] = train_df[state_cols].astype(str).agg("|".join, axis=1)

state_encoder = LabelEncoder()
train_df["state_id"] = state_encoder.fit_transform(train_df["state"])

# Encode actions
action_encoder = LabelEncoder()
train_df["action_id"] = action_encoder.fit_transform(train_df["action"])

joblib.dump(state_encoder, "state_encoder.pkl")
joblib.dump(action_encoder, "action_encoder.pkl")

print("Rows:", len(train_df))
print("States:", train_df["state_id"].nunique())
print("Actions:", train_df["action_id"].nunique())

Rows: 12935
States: 1011
Actions: 335


In [88]:
print(action_encoder.classes_[:20])

['West Africa|First Class' 'West Africa|Same Day'
 'West Africa|Second Class' 'West Africa|Standard Class']


# Build Action Lookup Table

In [89]:
from sklearn.preprocessing import LabelEncoder
import joblib
import pandas as pd
import numpy as np

# =========================
# Load split files
# =========================
train_df = pd.read_csv("dataco_rl_train_discrete.csv")
# Load non-discrete versions for val and test, then apply binning
val_df = pd.read_csv("dataco_rl_validation.csv")
test_df = pd.read_csv("dataco_rl_test.csv")

# =========================
# Load candidate fulfillment regions
# =========================
candidate_regions = pd.read_csv("candidate_fulfillment_regions.csv")

# Rename fulfillment region column if needed
if "Order Region" in candidate_regions.columns:
    candidate_regions = candidate_regions.rename(
        columns={"Order Region": "Fulfillment_Region"}
    )

# Keep only needed columns
candidate_regions = candidate_regions[
    ["Product Category Id", "Fulfillment_Region"]
].drop_duplicates()

# =========================
# Merge fulfillment regions into train/val/test
# =========================
train_df = train_df.merge(
    candidate_regions,
    on="Product Category Id",
    how="left"
)

val_df = val_df.merge(
    candidate_regions,
    on="Product Category Id",
    how="left"
)

test_df = test_df.merge(
    candidate_regions,
    on="Product Category Id",
    how="left"
)

# =========================
# Restore shipping mode text
# =========================
shipping_mode_map_reverse = {
    1: "Standard Class",
    2: "Second Class",
    3: "First Class",
    4: "Same Day"
}

for df_iter in [train_df, val_df, test_df]:
    if "Shipping Mode_str" not in df_iter.columns:
        df_iter["Shipping Mode_str"] = df_iter["shipping_mode_speed"].map(
            shipping_mode_map_reverse
        )

# =========================
# Restore order region text
# =========================
for df_iter in [train_df, val_df, test_df]:
    if "Order Region_str" not in df_iter.columns:
        df_iter["Order Region_str"] = "West Africa"
    # Add Order Region_bin manually, as it's a constant for West Africa
    df_iter['Order Region_bin'] = 0

# =========================
# Re-apply binning to val_df and test_df after loading non-discrete versions
# This ensures _bin columns are present for state creation.
# =========================
bin_edges = joblib.load("state_bins.pkl")
continuous_features = list(bin_edges.keys())

for df_to_bin in [val_df, test_df]:
    for col in continuous_features:
        if col in df_to_bin.columns:
            # Only bin if there are actual bins (more than just min/max)
            if len(bin_edges[col]) > 2:
                df_to_bin[col + "_bin"] = np.digitize(
                    df_to_bin[col],
                    bin_edges[col][1:-1]
                )
            else:
                # If only one bin, assign 0 (or a single fixed value for that bin)
                df_to_bin[col + "_bin"] = 0
        # If a continuous feature is missing from the DataFrame, ensure a corresponding _bin column is also handled
        # For example, if 'Shipping Cost' is missing, but 'Shipping Cost_bin' is in state_cols, this needs a default.
        # Given the previous context, all state_cols should originate from existing features.

# =========================
# Create route
# =========================
for df_iter in [train_df, val_df, test_df]:
    df_iter["Route"] = (
        df_iter["Order Region_str"].astype(str)
        + " -> "
        + df_iter["Order Country"].astype(str)
    )

# =========================
# Create combined action
# Fulfillment Region + Route + Shipping Mode
# =========================
for df_iter in [train_df, val_df, test_df]:
    df_iter["action"] = (
        df_iter["Fulfillment_Region"].astype(str)
        + " | "
        + df_iter["Route"].astype(str)
        + " | "
        + df_iter["Shipping Mode_str"].astype(str)
    )

# =========================
# Create state IDs
# =========================
state_cols = [
    "Product Category Id_bin",
    "stocking_score_bin",
    "margin_pct_bin",
    "shipping_delay_bin",
    "Order Region_bin", # Include this now as it's been set to 0
    "Shipping Cost_bin",
    "Order Item Total_bin"
]

# Check for missing state columns in train_df first
missing_train_state_cols = [col for col in state_cols if col not in train_df.columns]
if missing_train_state_cols:
    raise ValueError(f"Missing state columns in train_df: {missing_train_state_cols}")

train_df["state"] = train_df[state_cols].astype(str).agg("|".join, axis=1)

state_encoder = LabelEncoder()
train_df["state_id"] = state_encoder.fit_transform(train_df["state"])

# =========================
# Create action IDs
# =========================
action_encoder = LabelEncoder()
train_df["action_id"] = action_encoder.fit_transform(train_df["action"])

# Handle unseen actions/states in validation/test
valid_actions = set(action_encoder.classes_)
valid_states = set(state_encoder.classes_)

# Process val_df and test_df for state and action IDs
for df_proc in [val_df, test_df]:
    # First, create the 'state' column using the binned features
    missing_proc_state_cols = [col for col in state_cols if col not in df_proc.columns]
    if missing_proc_state_cols:
        # This is a critical check to ensure binning worked for all required state_cols
        # If a column was not binned because it wasn't in continuous_features or was all one value,
        # and it's needed for state_cols, a default value might be required.
        raise ValueError(f"Missing state columns in processed dataframe: {missing_proc_state_cols}")

    df_proc["state"] = df_proc[state_cols].astype(str).agg("|".join, axis=1)

    # Filter to include only actions and states seen in training
    df_proc = df_proc[df_proc["action"].isin(valid_actions)].copy()
    df_proc = df_proc[df_proc["state"].isin(valid_states)].copy()

    df_proc["action_id"] = action_encoder.transform(df_proc["action"])
    df_proc["state_id"] = state_encoder.transform(df_proc["state"])

# IMPORTANT: Update the global val_df and test_df after filtering/processing within the loop
# This is done by re-assigning here. This is a common pattern to avoid `SettingWithCopyWarning`
# and ensures the external dataframes are updated.
# A better approach would be to collect the processed dataframes and re-assign after the loop.
# Let's adjust for correct assignment.

# Collect processed dataframes
processed_val_df = None
processed_test_df = None

# Re-process val_df and test_df correctly to update external variables
val_df["state"] = val_df[state_cols].astype(str).agg("|".join, axis=1)
val_df = val_df[val_df["action"].isin(valid_actions)].copy()
val_df = val_df[val_df["state"].isin(valid_states)].copy()
val_df["action_id"] = action_encoder.transform(val_df["action"])
val_df["state_id"] = state_encoder.transform(val_df["state"])

test_df["state"] = test_df[state_cols].astype(str).agg("|".join, axis=1)
test_df = test_df[test_df["action"].isin(valid_actions)].copy()
test_df = test_df[test_df["state"].isin(valid_states)].copy()
test_df["action_id"] = action_encoder.transform(test_df["action"])
test_df["state_id"] = state_encoder.transform(test_df["state"])

# =========================
# Save encoders
# =========================
joblib.dump(state_encoder, "state_encoder.pkl")
joblib.dump(action_encoder, "action_encoder.pkl")

# =========================
# Final checks
# =========================
n_states = train_df["state_id"].nunique()
n_actions = train_df["action_id"].nunique()

print("Rows:", len(train_df))
print("States:", n_states)
print("Actions:", n_actions)

print("\nSample actions:")
print(train_df["action"].head(10))

print("\nAction encoder classes:")
print(action_encoder.classes_[:10])

Rows: 12935
States: 1011
Actions: 335

Sample actions:
0    Central America | West Africa -> Ghana | Secon...
1    Western Europe | West Africa -> Ghana | Second...
2    South America | West Africa -> Ghana | Second ...
3    Southern Europe | West Africa -> Ghana | Secon...
4    Northern Europe | West Africa -> Ghana | Secon...
5    Central America | West Africa -> Ghana | Secon...
6    Western Europe | West Africa -> Ghana | Second...
7    South America | West Africa -> Ghana | Second ...
8    Northern Europe | West Africa -> Ghana | Secon...
9    Southern Europe | West Africa -> Ghana | Secon...
Name: action, dtype: object

Action encoder classes:
['Caribbean | West Africa -> BenÃ\x83Â\xadn | First Class'
 'Caribbean | West Africa -> BenÃ\x83Â\xadn | Second Class'
 'Caribbean | West Africa -> BenÃ\x83Â\xadn | Standard Class'
 'Caribbean | West Africa -> Burkina Faso | Standard Class'
 'Caribbean | West Africa -> Costa de Marfil | First Class'
 'Caribbean | West Africa -> Costa de Mar

In [90]:
print("Rows:", len(train_df))
print("States:", train_df["state_id"].nunique())
print("Actions:", train_df["action_id"].nunique())
print(train_df["action"].head())

Rows: 12935
States: 1011
Actions: 335
0    Central America | West Africa -> Ghana | Secon...
1    Western Europe | West Africa -> Ghana | Second...
2    South America | West Africa -> Ghana | Second ...
3    Southern Europe | West Africa -> Ghana | Secon...
4    Northern Europe | West Africa -> Ghana | Secon...
Name: action, dtype: object


In [91]:
print("Q shape should be:")
print("States:", train_df["state_id"].nunique())
print("Actions:", train_df["action_id"].nunique())

n_states = train_df["state_id"].nunique()
n_actions = train_df["action_id"].nunique()

Q = np.zeros((n_states, n_actions))

print(Q.shape)

Q shape should be:
States: 1011
Actions: 335
(1011, 335)


# Check State Space Size

In [92]:
print("States:", train_df["state_id"].nunique())
print("Actions:", train_df["action_id"].nunique())

States: 1011
Actions: 335


In [93]:
print(
    "Unique states:",
    train_df['state'].nunique()
)

print(
    "Unique actions:",
    train_df['action'].nunique()
)

print(
    "Q-table size:",
    train_df['state'].nunique()
    *
    train_df['action'].nunique()
)

Unique states: 1011
Unique actions: 335
Q-table size: 338685


# **STEP 6: Q-LEARNING IMPLEMENTATION**


In [98]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

# ====================================
# Load training data
# ====================================
train_df = pd.read_csv("dataco_rl_train_discrete.csv")

# ====================================
# Restore fulfillment regions
# ====================================
candidate_regions = pd.read_csv("candidate_fulfillment_regions.csv")

if "Order Region" in candidate_regions.columns:
    candidate_regions = candidate_regions.rename(
        columns={"Order Region": "Fulfillment_Region"}
    )

candidate_regions = candidate_regions[
    ["Product Category Id", "Fulfillment_Region"]
].drop_duplicates()

train_df = train_df.merge(
    candidate_regions,
    on="Product Category Id",
    how="left"
)

# ====================================
# Restore shipping mode text
# ====================================
shipping_mode_map_reverse = {
    1: "Standard Class",
    2: "Second Class",
    3: "First Class",
    4: "Same Day"
}

train_df["Shipping Mode_str"] = (
    train_df["shipping_mode_speed"]
    .map(shipping_mode_map_reverse)
)

# ====================================
# Restore region text
# ====================================
train_df["Order Region_str"] = "West Africa"

# ====================================
# Create route
# ====================================
train_df["Route"] = (
    train_df["Order Region_str"].astype(str)
    + " -> "
    + train_df["Order Country"].astype(str)
)

# ====================================
# Create ACTION
# Fulfillment Region + Route + Shipping Mode
# ====================================
train_df["action"] = (
    train_df["Fulfillment_Region"].astype(str)
    + " | "
    + train_df["Route"].astype(str)
    + " | "
    + train_df["Shipping Mode_str"].astype(str)
)

# ====================================
# Create STATE
# ====================================
state_cols = [
    "Product Category Id_bin",
    "stocking_score_bin",
    "margin_pct_bin",
    "shipping_delay_bin",
    "Shipping Cost_bin",
    "Order Item Total_bin"
]

train_df["state"] = (
    train_df[state_cols]
    .astype(str)
    .agg("|".join, axis=1)
)

# ====================================
# Encode state/action
# ====================================
state_encoder = LabelEncoder()
action_encoder = LabelEncoder()

train_df["state_id"] = state_encoder.fit_transform(
    train_df["state"]
)

train_df["action_id"] = action_encoder.fit_transform(
    train_df["action"]
)

print("Rows:", len(train_df))
print("States:", train_df["state_id"].nunique())
print("Actions:", train_df["action_id"].nunique())

print("\nSample actions:")
print(train_df["action"].head())

print("\nEncoder classes:")
print(action_encoder.classes_[:10])

Rows: 12935
States: 1011
Actions: 335

Sample actions:
0    Central America | West Africa -> Ghana | Secon...
1    Western Europe | West Africa -> Ghana | Second...
2    South America | West Africa -> Ghana | Second ...
3    Southern Europe | West Africa -> Ghana | Secon...
4    Northern Europe | West Africa -> Ghana | Secon...
Name: action, dtype: object

Encoder classes:
['Caribbean | West Africa -> BenÃ\x83Â\xadn | First Class'
 'Caribbean | West Africa -> BenÃ\x83Â\xadn | Second Class'
 'Caribbean | West Africa -> BenÃ\x83Â\xadn | Standard Class'
 'Caribbean | West Africa -> Burkina Faso | Standard Class'
 'Caribbean | West Africa -> Costa de Marfil | First Class'
 'Caribbean | West Africa -> Costa de Marfil | Same Day'
 'Caribbean | West Africa -> Costa de Marfil | Second Class'
 'Caribbean | West Africa -> Costa de Marfil | Standard Class'
 'Caribbean | West Africa -> Ghana | First Class'
 'Caribbean | West Africa -> Ghana | Second Class']


# Dual-Goal Reward Fuction

In [118]:
PROFIT_WEIGHT = 5.0
LATE_PENALTY = 8.0
TIME_PENALTY = 0.10

def calculate_reward(row):
    return (
        PROFIT_WEIGHT * row["Order Profit Per Order"]
        - LATE_PENALTY * row["Late_delivery_risk"]
        - TIME_PENALTY * row["Days for shipping (real)"]
    )

train_df["reward"] = train_df.apply(calculate_reward, axis=1)

n_states = train_df["state_id"].nunique()
n_actions = train_df["action_id"].nunique()

# Initialize Q-Table and Train

In [119]:
Q = np.zeros((n_states, n_actions))

alpha = 0.1
gamma = 0.5
epsilon = 1.0
epsilon_min = 0.05
epsilon_decay = 0.995
episodes = 100

reward_table = (
    train_df.groupby(["state_id", "action_id"])["reward"]
    .mean()
    .to_dict()
)

state_ids = train_df["state_id"].to_numpy()
mean_reward = train_df["reward"].mean()

for episode in range(episodes):
    total_reward = 0

    for i in range(len(state_ids) - 1):
        state = int(state_ids[i])
        next_state = int(state_ids[i + 1])

        if np.random.rand() < epsilon:
            action = np.random.randint(n_actions)
        else:
            action = np.argmax(Q[state])

        reward = reward_table.get((state, action), mean_reward)

        Q[state, action] += alpha * (
            reward + gamma * np.max(Q[next_state]) - Q[state, action]
        )

        total_reward += reward

    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    if episode % 10 == 0:
        print(f"Episode {episode}: Reward = {total_reward:.2f}")

print("Training complete")

Episode 0: Reward = 226198.99
Episode 10: Reward = 226217.56
Episode 20: Reward = 226535.61
Episode 30: Reward = 226927.51
Episode 40: Reward = 227305.94
Episode 50: Reward = 228210.37
Episode 60: Reward = 228447.59
Episode 70: Reward = 228777.39
Episode 80: Reward = 229085.88
Episode 90: Reward = 230077.18
Training complete


In [120]:
policy = np.argmax(Q, axis=1)

policy_df = pd.DataFrame({
    "state_id": np.arange(n_states),
    "recommended_action": action_encoder.inverse_transform(policy)
})

policy_df.head(10)

,state_id,recommended_action
0,0,Southeast Asia | West Africa -> Mauritania | S...
1,1,Oceania | West Africa -> NÃÂ­ger | First Class
2,2,Central America | West Africa -> Senegal | Sam...
3,3,West of USA | West Africa -> Senegal | First ...
4,4,Central America | West Africa -> Guinea-Bissau...
5,5,Southern Europe | West Africa -> Mali | Standa...
6,6,Northern Europe | West Africa -> Mauritania | ...
7,7,Oceania | West Africa -> Togo | Standard Class
8,8,Southern Europe | West Africa -> Nigeria | Sec...
9,9,Western Europe | West Africa -> Ghana | First ...


# Extract the Learned Policy

In [121]:
policy_action_ids = np.argmax(Q, axis=1)

policy_df = pd.DataFrame({
    "state_id": np.arange(Q.shape[0]),
    "action_id": policy_action_ids,
    "recommended_action": action_encoder.inverse_transform(policy_action_ids)
})

# Split combined action into readable columns
policy_df[
    [
        "Recommended_Fulfillment_Region",
        "Recommended_Route",
        "Recommended_Shipping_Mode"
    ]
] = policy_df["recommended_action"].str.split(" \\| ", expand=True)

policy_df.head(10)

,state_id,action_id,recommended_action,Recommended_Fulfillment_Region,Recommended_Route,Recommended_Shipping_Mode
0,0,223,Southeast Asia | West Africa -> Mauritania | S...,Southeast Asia,West Africa -> Mauritania,Standard Class
1,1,147,Oceania | West Africa -> NÃÂ­ger | First Class,Oceania,West Africa -> NÃÂ­ger,First Class
2,2,68,Central America | West Africa -> Senegal | Sam...,Central America,West Africa -> Senegal,Same Day
3,3,287,West of USA | West Africa -> Senegal | First ...,West of USA,West Africa -> Senegal,First Class
4,4,47,Central America | West Africa -> Guinea-Bissau...,Central America,West Africa -> Guinea-Bissau,First Class
5,5,254,Southern Europe | West Africa -> Mali | Standa...,Southern Europe,West Africa -> Mali,Standard Class
6,6,102,Northern Europe | West Africa -> Mauritania | ...,Northern Europe,West Africa -> Mauritania,Standard Class
7,7,159,Oceania | West Africa -> Togo | Standard Class,Oceania,West Africa -> Togo,Standard Class
8,8,259,Southern Europe | West Africa -> Nigeria | Sec...,Southern Europe,West Africa -> Nigeria,Second Class
9,9,299,Western Europe | West Africa -> Ghana | First ...,Western Europe,West Africa -> Ghana,First Class


In [122]:
print(policy_df.head(10))
print(policy_df["recommended_action"].head(10))

   state_id  action_id                                 recommended_action  \
0         0        223  Southeast Asia | West Africa -> Mauritania | S...   
1         1        147    Oceania | West Africa -> NÃÂ­ger | First Class   
2         2         68  Central America | West Africa -> Senegal | Sam...   
3         3        287  West of USA  | West Africa -> Senegal | First ...   
4         4         47  Central America | West Africa -> Guinea-Bissau...   
5         5        254  Southern Europe | West Africa -> Mali | Standa...   
6         6        102  Northern Europe | West Africa -> Mauritania | ...   
7         7        159     Oceania | West Africa -> Togo | Standard Class   
8         8        259  Southern Europe | West Africa -> Nigeria | Sec...   
9         9        299  Western Europe | West Africa -> Ghana | First ...   

  Recommended_Fulfillment_Region             Recommended_Route  \
0                 Southeast Asia     West Africa -> Mauritania   
1                   

In [123]:
policy_df.to_csv("learned_policy.csv", index=False)
print("Saved learned_policy.csv")

Saved learned_policy.csv


# Evaluate Recommended Shipping Modes

# Measure Improvement

# Save Results

In [114]:
policy_df.to_csv(
    "q_learning_policy.csv",
    index=False
)

np.save(
    "q_table.npy",
    Q
)

print("Policy and Q-table saved")

Policy and Q-table saved


# **STEP 7: SARSA IMPLEMENTATION**


# **STEP 8: COMPARISON OF PROFIT, REWARD, AND ON-TIME DELIVERY PERFORMANCE**


* State = information known BEFORE shipment decision
* Action = shipping method / route / priority decision
* Reward = profit − lateness penalty
* Next state = next order environment

Reward Function

Rt=α(Profit)−β(Late Shipment Penalty)

Where:

α controls profit importance

β controls delivery performance importance

A more detailed version:

Rt=αPt−βLt−γCt

Where:
Pt = profit

Lt = lateness indicator or delay days

Ct = shipping cost

# STEP 8: Q-learning or SARSA environment setup